In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from polypesto.core import create_sim_conditions, simulate_problem
from polypesto.core.pypesto import calculate_cis, create_ensemble, predict_with_ensemble
from polypesto.examples.base import output_dirs

# Model specific imports
from polypesto.models.binary import BinaryIrreversible
from polypesto.models.binary.utils import create_ensemble_pred_problem
from polypesto.vis import plot_ensemble_predictions
from polypesto.core.study import Study

%load_ext autoreload
%autoreload 2

In [3]:
Study.load("jobs/outputs/best_obs/obs_all")

FileNotFoundError: File not found: jobs/outputs/best_obs/obs_all/model_config.json

In [2]:
dir_all = "jobs/outputs/best_obs/obs_all"
model_all = BinaryIrreversible(
    observables=["FA", "FB", "fA", "fB", "xA", "xB"], obs_noise=0.01
)

study_all = Study.load(dir_all, model_all)

In [3]:
study_all.get_ensembles()

{'prob_000 | p_000': <pypesto.ensemble.ensemble.Ensemble at 0x125e238c0>,
 'prob_001 | p_000': <pypesto.ensemble.ensemble.Ensemble at 0x125e2fed0>,
 'prob_002 | p_000': <pypesto.ensemble.ensemble.Ensemble at 0x125fa8690>,
 'prob_003 | p_000': <pypesto.ensemble.ensemble.Ensemble at 0x125fa48a0>,
 'prob_004 | p_000': <pypesto.ensemble.ensemble.Ensemble at 0x125fa49d0>,
 'prob_000 | p_001': <pypesto.ensemble.ensemble.Ensemble at 0x125fc68d0>,
 'prob_001 | p_001': <pypesto.ensemble.ensemble.Ensemble at 0x125ecf460>,
 'prob_002 | p_001': <pypesto.ensemble.ensemble.Ensemble at 0x125ecfac0>,
 'prob_003 | p_001': <pypesto.ensemble.ensemble.Ensemble at 0x126290850>,
 'prob_004 | p_001': <pypesto.ensemble.ensemble.Ensemble at 0x126290c50>,
 'prob_000 | p_002': <pypesto.ensemble.ensemble.Ensemble at 0x125eda8a0>,
 'prob_001 | p_002': <pypesto.ensemble.ensemble.Ensemble at 0x125edb2f0>,
 'prob_002 | p_002': <pypesto.ensemble.ensemble.Ensemble at 0x126295d30>,
 'prob_003 | p_002': <pypesto.ensemble

In [4]:
summaries_all = {}

for key, ens in study_all.ensembles.items():
    if ens.x_vectors.size != 0:
        summaries_all[key] = ens.compute_summary(percentiles_list=(5, 25, 75, 95))

In [5]:
summaries_all

{'prob_000 | p_000': {'mean': array([0.09141837, 0.09981166]),
  'std': array([0.15163994, 0.00250487]),
  'median': array([0.02347816, 0.09956512]),
  'percentile 5': array([0.0013169 , 0.09637867]),
  'percentile 25': array([0.00474814, 0.09818108]),
  'percentile 75': array([0.10168411, 0.10100132]),
  'percentile 95': array([0.41474668, 0.10453751])},
 'prob_001 | p_000': {'mean': array([0.06150791, 0.09523024]),
  'std': array([0.05027977, 0.00649368]),
  'median': array([0.05397514, 0.09418736]),
  'percentile 5': array([0.00215312, 0.0865835 ]),
  'percentile 25': array([0.01543879, 0.09000917]),
  'percentile 75': array([0.09611453, 0.09969793]),
  'percentile 95': array([0.15586893, 0.10739416])},
 'prob_002 | p_000': {'mean': array([0.12636381, 0.12681483]),
  'std': array([1.72708626, 1.73698901]),
  'median': array([0.00384217, 0.00393485]),
  'percentile 5': array([0.00114889, 0.00116139]),
  'percentile 25': array([0.0020675 , 0.00206389]),
  'percentile 75': array([0.007

In [13]:
dir_A = "jobs/outputs/best_obs/obs_A"
model_A = BinaryIrreversible(observables=["FA", "fA", "xA"], obs_noise=0.01)

study_A = Study.load(dir_A, model_A)

In [14]:
summaries_A = {}

for key, ens in study_A.ensembles.items():
    if ens.x_vectors.size != 0:
        summaries_A[key] = ens.compute_summary(percentiles_list=(5, 25, 75, 95))
        print(ens.check_identifiability())

            parameterId  lowerBound  upperBound  ensemble_mean  ensemble_std  \
parameterId                                                                    
rA                   rA       0.001       100.0       0.120863      0.189793   
rB                   rB       0.001       100.0       0.100267      0.002881   

             ensemble_median  within lb: 1 std  within ub: 1 std  \
parameterId                                                        
rA                  0.120863             False              True   
rB                  0.100267              True              True   

             within lb: 2 std  within ub: 2 std  within lb: 3 std  \
parameterId                                                         
rA                      False              True             False   
rB                       True              True              True   

             within ub: 3 std  within lb: perc 5  within lb: perc 20  \
parameterId                                              

In [15]:
for key, val in summaries_all.items():
    if key in summaries_A:
        print("key: ", key)
        print("summary all: ", val)
        print("summary A: ", summaries_A[key])

key:  prob_000 | p_000
summary all:  {'mean': array([0.09141837, 0.09981166]), 'std': array([0.15163994, 0.00250487]), 'median': array([0.02347816, 0.09956512]), 'percentile 5': array([0.0013169 , 0.09637867]), 'percentile 25': array([0.00474814, 0.09818108]), 'percentile 75': array([0.10168411, 0.10100132]), 'percentile 95': array([0.41474668, 0.10453751])}
summary A:  {'mean': array([0.12086262, 0.10026664]), 'std': array([0.18979317, 0.00288082]), 'median': array([0.03234307, 0.09988758]), 'percentile 5': array([0.00136315, 0.09640891]), 'percentile 25': array([0.00535336, 0.09826952]), 'percentile 75': array([0.15076852, 0.10166235]), 'percentile 95': array([0.54097675, 0.10587418])}
key:  prob_001 | p_000
summary all:  {'mean': array([0.06150791, 0.09523024]), 'std': array([0.05027977, 0.00649368]), 'median': array([0.05397514, 0.09418736]), 'percentile 5': array([0.00215312, 0.0865835 ]), 'percentile 25': array([0.01543879, 0.09000917]), 'percentile 75': array([0.09611453, 0.0996

In [16]:
# Test results_summary() at the study level
study_summary = study_all.results_summary()

print("Study-level summary:")
print(study_summary.head(10))
print("\nShape:", study_summary.shape)
print("\nIndex levels:", study_summary.index.names)
print("\nColumns:", study_summary.columns.tolist())

Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Study-level summary:
                             parameterId  lowerBound  upperBound  \
problem_key      parameterId                                       
prob_000 | p_000 rA                   rA       0.001       100.0   
                 rB                   rB       0.001       100.0   
prob_001 | p_000 rA                   rA       0.001       100.0   
                 rB                   rB       0.001       100.0   
prob_002 | p_000 rA                   rA       0.001       100.0   
                 rB                   rB       0.001       100.0   
prob_003 | p_000 rA                   rA       0.001       100.0   
                 rB                   rB       0.001       100.0   
prob_004 | p_000 rA                   rA       0.001       100.0   
                 rB                   rB       0.001       100.0   

              

In [17]:
study_summary.to_csv("output.csv")

In [18]:
# Compare summary for study_all vs study_A
study_A_summary = study_A.results_summary()

print("Comparing rA parameter across studies:")
print("\nAll observables (study_all):")
rA_all = study_summary.xs("rA", level="parameterId")
print(rA_all[["ensemble_mean", "ensemble_std", "true_value"]].head())

print("\nA observables only (study_A):")
rA_A = study_A_summary.xs("rA", level="parameterId")
print(rA_A[["ensemble_mean", "ensemble_std", "true_value"]].head())

Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Ensemble is empty - cannot summarize
Comparing rA parameter across studies:

All observables (study_all):
                  ensemble_mean  ensemble_std  true_value
problem_key                                              
prob_000 | p_000       0.091418      0.151640         0.1
prob_001 | p_000       0.061508      0.050280         0.1
prob_002 | p_000       0.126364      1.727086         0.1
prob_003 | p_000       0.095520      0.006486         0.1
prob_004 | p_000       0.099806      0.002457         0.1

A observables only (study_A):
                  ensemble_mean  ensemble_std  true_value
problem_key                                              
prob_000 | p_000       0.120863      0.189793         0.1
prob_001 | p_000       0.055430      0.054997         0.1
prob_002 | p_000       0.090560      1.0

In [ ]:
# Get data for a specific problem
specific_problem = study_summary.loc["prob_000 | p_000"]
print("Summary for problem prob_000 | p_000:")
print(specific_problem[["ensemble_mean", "ensemble_std", "true_value"]])

In [ ]:
# Flatten for easier filtering
study_summary_flat = study_summary.reset_index()

print("Flattened DataFrame:")
print(
    study_summary_flat[
        ["problem_key", "parameterId", "ensemble_mean", "ensemble_std", "true_value"]
    ].head(10)
)